# 06 — Hybrid Recommendation System

**Goal:** combine content similarity and collaborative evidence with configurable weights. Each component is scaled to a comparable 0–1 range before blending.

## 1. Load data and saved model artifacts

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Keep paths consistent whether the notebook is launched from the project root or 02_Notebooks.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "02_Notebooks" else CWD
DATA_RAW = PROJECT_ROOT / "01_Data" / "Raw"
DATA_PROCESSED = PROJECT_ROOT / "01_Data" / "Processed"
MODELS = PROJECT_ROOT / "03_Models"
VISUALS = PROJECT_ROOT / "05_Visualizations"
for folder in [DATA_RAW, DATA_PROCESSED, MODELS, VISUALS]:
    folder.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

import joblib
from scipy.sparse import csr_matrix
movies = pd.read_pickle(DATA_PROCESSED / "movies_processed.pkl").reset_index(drop=True)
ratings = pd.read_csv(DATA_PROCESSED / "ratings_processed.csv")
content = joblib.load(MODELS / "content_artifacts.joblib")
collab = joblib.load(MODELS / "collaborative_artifacts.joblib")
content_matrix = content["content_matrix"]
content_nn = content["content_nn"]
user_item = collab["user_item"]
item_similarity = collab["item_similarity"]
user_to_idx = collab["user_to_idx"]
rated_movie_ids = collab["rated_movie_ids"]
movie_to_idx = collab["movie_to_idx"]

Project root: D:\DataScience\notebooks\CineMatch_Ready_Notebooks


## 2. Helper functions for component scores

Content scores come from similarity to the user's positively rated movies. Collaborative scores are based on item-item rating evidence. A min-max function scales each candidate list to 0–1.

In [2]:
def minmax_dict(score_dict):
    if not score_dict:
        return {}
    vals = np.array(list(score_dict.values()), dtype=float)
    lo, hi = vals.min(), vals.max()
    if np.isclose(lo, hi):
        return {k: 1.0 for k in score_dict}
    return {k: float((v-lo)/(hi-lo)) for k, v in score_dict.items()}

def get_seen_movie_ids(user_id):
    if int(user_id) not in user_to_idx:
        return set()
    uidx = user_to_idx[int(user_id)]
    start, end = user_item.indptr[uidx], user_item.indptr[uidx+1]
    return set(int(rated_movie_ids[c]) for c in user_item.indices[start:end])

def content_scores_from_history(user_id, limit_pool=500):
    if int(user_id) not in user_to_idx:
        return {}
    uidx = user_to_idx[int(user_id)]
    start, end = user_item.indptr[uidx], user_item.indptr[uidx+1]
    cols, vals = user_item.indices[start:end], user_item.data[start:end]
    liked_cols = [c for c, r in zip(cols, vals) if r >= 4.0]
    if not liked_cols:
        liked_cols = list(cols)
    scores = {}
    for col in liked_cols:
        mid = int(rated_movie_ids[col])
        movie_row = movies.index[movies.movieId == mid]
        if len(movie_row) == 0: 
            continue
        distances, indices = content_nn.kneighbors(content_matrix[int(movie_row[0])],
                                                    n_neighbors=min(limit_pool+1, len(movies)))
        for dist, idx in zip(distances[0], indices[0]):
            candidate_id = int(movies.iloc[int(idx)].movieId)
            scores[candidate_id] = max(scores.get(candidate_id, 0.0), float(1-dist))
    return scores

def collaborative_scores_for_user(user_id):
    if int(user_id) not in user_to_idx:
        return {}
    uidx = user_to_idx[int(user_id)]
    start, end = user_item.indptr[uidx], user_item.indptr[uidx+1]
    cols, vals = user_item.indices[start:end], user_item.data[start:end]
    scores = {}
    for col, rating_value in zip(cols, vals):
        similarities = item_similarity[:, col].tocoo()
        for candidate_col, sim in zip(similarities.row, similarities.data):
            if sim > 0:
                candidate_id = int(rated_movie_ids[candidate_col])
                scores[candidate_id] = scores.get(candidate_id, 0.0) + float(sim * rating_value)
    return scores

## 3. Build the weighted hybrid recommender

Default weights are 0.4 content and 0.6 collaborative, as an initial configuration—not a claim that these weights are optimal. Evaluation should guide tuning.

In [3]:
def hybrid_recommendations(user_id, top_n=10, content_weight=0.4, collaborative_weight=0.6,
                           fallback_popular=True):
    if content_weight < 0 or collaborative_weight < 0:
        raise ValueError("Weights must be non-negative.")
    total = content_weight + collaborative_weight
    if total == 0:
        raise ValueError("At least one weight must be greater than zero.")
    cw, rw = content_weight/total, collaborative_weight/total

    seen = get_seen_movie_ids(user_id)
    c_scores = minmax_dict(content_scores_from_history(user_id))
    r_scores = minmax_dict(collaborative_scores_for_user(user_id))
    candidate_ids = set(c_scores) | set(r_scores)
    combined = {}
    for mid in candidate_ids - seen:
        combined[mid] = cw*c_scores.get(mid, 0.0) + rw*r_scores.get(mid, 0.0)

    if not combined and fallback_popular:
        popular = movies.sort_values(["rating_count","average_rating"], ascending=False)
        for row in popular.itertuples():
            if int(row.movieId) not in seen:
                combined[int(row.movieId)] = float(row.average_rating)
            if len(combined) >= top_n:
                break

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_n]
    result = pd.DataFrame(ranked, columns=["movieId","hybrid_score"])
    return result.merge(movies[["movieId","title","genres","average_rating","rating_count"]],
                        on="movieId", how="left")

## 4. Test users and weights

Compare the output after changing the weights. The list may change because the relative contribution of each recommendation source changes.

In [4]:
known_user = int(collab["user_ids"][0])
print("User:", known_user, " | default 40/60 blend")
display(hybrid_recommendations(known_user, top_n=10))
print("Content-heavy 70/30 blend")
display(hybrid_recommendations(known_user, top_n=10, content_weight=0.7, collaborative_weight=0.3))

User: 1  | default 40/60 blend


,movieId,hybrid_score,title,genres,average_rating,rating_count
0,858,0.646430,"Godfather, The (1972)",Crime|Drama,4.289062,192
1,1484,0.636200,"Daytrippers, The (1996)",Comedy|Drama|Mystery|Romance,3.666667,3
2,2889,0.630111,"Mystery, Alaska (1999)",Comedy|Drama,3.750000,8
3,2968,0.616991,Time Bandits (1981),Adventure|Comedy|Fantasy|Sci-Fi,3.731707,41
4,3327,0.612596,Beyond the Mat (1999),Documentary,3.700000,5
5,4855,0.603818,Dirty Harry (1971),Action|Crime|Thriller,3.775000,20
6,4571,0.601764,Bill & Ted's Excellent Adventure (1989),Adventure|Comedy|Sci-Fi,3.700000,45
7,1221,0.593934,"Godfather: Part II, The (1974)",Crime|Drama,4.259690,129
8,78499,0.592271,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,4.109091,55
9,6832,0.586984,Regarding Henry (1991),Drama,3.600000,5


Content-heavy 70/30 blend


,movieId,hybrid_score,title,genres,average_rating,rating_count
0,1407,0.588434,Scream (1996),Comedy|Horror|Mystery|Thriller,3.200000,70
1,2587,0.581351,Life (1999),Comedy|Crime|Drama,3.250000,4
2,2011,0.566229,Back to the Future Part II (1989),Adventure|Comedy|Sci-Fi,3.505747,87
3,3051,0.554564,Anywhere But Here (1999),Comedy|Drama,2.500000,3
4,2968,0.541056,Time Bandits (1981),Adventure|Comedy|Fantasy|Sci-Fi,3.731707,41
5,2001,0.538177,Lethal Weapon 2 (1989),Action|Comedy|Crime|Drama,3.180851,47
6,2471,0.532763,Crocodile Dundee II (1988),Action|Adventure|Comedy,2.479167,24
7,4571,0.532071,Bill & Ted's Excellent Adventure (1989),Adventure|Comedy|Sci-Fi,3.700000,45
8,2485,0.531283,She's All That (1999),Comedy|Romance,2.844828,29
9,3438,0.530048,Teenage Mutant Ninja Turtles (1990),Action|Children|Comedy|Fantasy|Sci-Fi,3.000000,25


## 5. Cold-start handling

A new user has no rating history, so personalised content/collaborative evidence is unavailable. The fallback below recommends popular movies. A new movie can still participate in content recommendations when its metadata is added and the content model is rebuilt.

In [5]:
new_user_id = -999999
display(hybrid_recommendations(new_user_id, top_n=10))
print("For a new movie: add its metadata to the movie catalogue and refit TF-IDF/nearest-neighbour artifacts.")

,movieId,hybrid_score,title,genres,average_rating,rating_count
0,318,4.429022,"Shawshank Redemption, The (1994)",Crime|Drama,4.429022,317
1,260,4.231076,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.231076,251
2,527,4.225000,Schindler's List (1993),Drama|War,4.225000,220
3,296,4.197068,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.197068,307
4,2571,4.192446,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.192446,278
5,356,4.164134,Forrest Gump (1994),Comedy|Drama|Romance|War,4.164134,329
6,593,4.161290,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.161290,279
7,110,4.031646,Braveheart (1995),Action|Drama|War,4.031646,237
8,589,3.970982,Terminator 2: Judgment Day (1991),Action|Sci-Fi,3.970982,224
9,480,3.750000,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,3.750000,238


For a new movie: add its metadata to the movie catalogue and refit TF-IDF/nearest-neighbour artifacts.


## 6. Save hybrid configuration and artifacts

In [6]:
hybrid_config = {"content_weight": 0.4, "collaborative_weight": 0.6,
                 "score_scaling": "min-max within each user's candidate list"}
joblib.dump(hybrid_config, MODELS / "hybrid_config.joblib")
print("Saved hybrid_config.joblib")

Saved hybrid_config.joblib


## Important limitation

Min-max scaling makes the component candidate scores comparable in range, but it does not guarantee calibrated probabilities. The weights are hyperparameters and should be selected using the evaluation protocol in Notebook 07.

Next: **07_Model_Evaluation.ipynb**.